> **QAC reproduction record** — claim `gidney-ekera-rsa2021`, reproduction `qsharp-modexp-qre-2026-11`.
>
> This notebook is the executable evidence behind the QAC assessment of Gidney and Ekerå (2021),
> *How to factor 2048 bit RSA integers in 8 hours using 20 million noisy qubits*, Quantum **5**, 433.
> It is kept readable on purpose: the markdown walks through the method so it can be used for learning.
> The **last cell writes `results.json`**, the only source of the numbers the public record attributes to QAC;
> nothing in that file is typed by hand. The environment is pinned in `requirements.lock.txt`,
> and `run.json` records who executed it and when. Re-execute with `qacclaims execute`.
> The author-code cross-check downloads one file from arXiv, so the notebook needs network access.

# Factoring RSA-2048 with windowed, coset-represented, runway-split arithmetic

Gidney and Ekerå estimate that a 2048-bit RSA integer can be factored in about 8 hours with about
20 million noisy qubits (title), on a planar grid with nearest-neighbour connectivity, a physical gate
error rate of $10^{-3}$, a 1 µs surface-code cycle and a 10 µs reaction time (abstract). Almost all of
the cost is one modular exponentiation, built from **lookup additions**. This notebook rebuilds that
exponentiation as a circuit and costs it:

1. **The circuit.** `modexp.qs` implements Section 2 of the paper in Q#: the coset representation of
   modular integers (Sec. 2.4), windowed arithmetic over exponent and factor bits (Sec. 2.5), oblivious
   carry runways with the runway reduction of Fig. 3 (Secs. 2.6–2.7), Cuccaro ripple-carry adders
   confined to each runway piece, table lookups with measurement-based uncomputation, and the
   semi-classical Fourier transform with qubit recycling (Sec. 2.3).
2. **Checks on small instances.** The adders, the whole modular exponentiation (exactly, against
   `pow(g, e, N)`), the initialization in superposition, and an end-to-end order-finding run whose
   measured frequencies are compared with the ideal distribution.
3. **The count at full size.** The circuit is counted with the QDK at $n = 2048$ with the parameters of
   Table 3.
4. **A cross-check with the authors' code.** The ancillary script that produced Tables 3–5 is
   downloaded from arXiv, checked against a pinned SHA-256, and run. It is a cross-check only: none of
   QAC's figures come from it, and its values appear only in this notebook's output.
5. **Physical cost.** The layout and timing model of Secs. 2.10–2.14, rebuilt as code
   (`physical_layout.py`), is applied to the circuit's counts. The QDK resource estimator gives an
   independent estimate.

## Parameters

Table 3 (p. 16), row $n = 2048$, fixes the construction: Ekerå–Håstad's algorithm needs
$n_e = 3(n/2 - 1) - 40$ exponent qubits for a single run; windows $c_\text{exp} = c_\text{mul} = 5$;
runway separation $c_\text{sep} = 1024$; padding offset $\delta_\text{off} = 4$ with
$c_\text{pad} = 2\lg n + \lg n_e + \delta_\text{off}$ (Secs. 2.8 and 3.1); level-1 and level-2 code
distances 15 and 27. The paper does not say how $c_\text{pad}$ is rounded; it must be an integer, so the
notebook rounds $2\lg n + \lg n_e$ up.

In [1]:
import math

N_BITS = 2048                          # n
N_E = 3 * (N_BITS // 2 - 1) - 40       # Table 3: n_e = 3(n/2 - 1) - 40
C_EXP, C_MUL = 5, 5                    # Table 3
C_SEP = 1024                           # Table 3
DELTA_OFF = 4                          # Table 3
C_PAD = math.ceil(2 * math.log2(N_BITS) + math.log2(N_E)) + DELTA_OFF   # Secs. 2.8, 3.1
LEVEL1_DISTANCE_TABLE3, LEVEL2_DISTANCE_TABLE3 = 15, 27                  # Table 3

# Physical assumptions (abstract, p. 1; Table 2, p. 3)
PHYSICAL_ERROR_RATE = 1e-3
CYCLE_TIME_NS = 1000
REACTION_TIME_NS = 10_000

PIECES = -(-N_BITS // C_SEP)
print(f"n_e = {N_E}, c_pad = {C_PAD}, runway pieces = {PIECES}")

n_e = 3029, c_pad = 38, runway pieces = 2


## The circuit

`modexp.qs` sits next to this notebook. A register is stored as pieces: piece $i$ holds $c_\text{sep}$ low
bits followed by $c_\text{pad}$ padding qubits — an oblivious carry runway for the lower pieces and the
coset padding for the top one. The register's value is $\sum_i \text{piece}_i \cdot 2^{i c_\text{sep}}$,
and the integer it represents is that value modulo $N$. Each windowed multiplication is two
multiply-adds, $y \mathrel{+}= x k$ and $x \mathrel{+}= y\,(-k^{-1})$, then a swap (Sec. 2.2). Each
multiply-add iterates over the factor's bits $c_\text{mul}$ at a time; each step looks up a classically
precomputed multiple of $k$ addressed by the exponent window and the factor window, adds it into every
piece of the target in parallel, and uncomputes the lookup.

The helpers below build the circuit's configuration and decode a measured register.

In [2]:
import random
import time
from collections import Counter
from fractions import Fraction
from pathlib import Path

import numpy as np
import qdk
from qdk import qsharp

qsharp.init()
qsharp.eval(Path("modexp.qs").read_text())
SEED = 2021
qsharp.set_quantum_seed(SEED)
qsharp.set_classical_seed(SEED)


def exponent_windows(ne, cexp):
    # Windows [lo, hi) from bit 0 upward; the circuit processes the most significant window first.
    return list(reversed([(lo, min(lo + cexp, ne)) for lo in range(0, ne, cexp)]))


def make_config(n, N, g, ne, cexp, cmul, csep, cpad, superpose, placeholder=False, cache=False, windows=None):
    ws = exponent_windows(ne, cexp)
    sizes = [hi - lo for lo, hi in ws] if windows is None else windows
    bases = [] if placeholder else [pow(g, 2 ** lo, N) for lo, hi in ws]
    return qdk.code.MakeConfig(n, csep, cpad, cmul, sizes, bases, N, superpose, placeholder, cache)


def piece_lens(n, csep):
    return [min(csep, n - i * csep) for i in range(-(-n // csep))]


def decode(bits, n, csep, cpad):
    value, pos = 0, 0
    for i, low in enumerate(piece_lens(n, csep)):
        size = low + cpad
        value += sum(1 << b for b, v in enumerate(bits[pos:pos + size]) if v) << (i * csep)
        pos += size
    return value


def run(cfg, quantum_exponent, exponent_bits, shots):
    # Shots run one at a time so that the seeded simulator gives the same results on every execution.
    return [qdk.code.RunModExp(cfg, quantum_exponent, exponent_bits) for _ in range(shots)]

## Checks on small instances

### 1. The adders

The paper's additions use the Cuccaro et al. ripple-carry adder (its Ref. [18]). The circuit uses two
variants: modulo $2^m$ inside each runway piece ($2(m-1)$ Toffolis), and with a carry-out for the runway
reduction of Fig. 3 ($2m$ Toffolis). Both are checked on random inputs; each must also restore its
addend.

In [3]:
rng = random.Random(SEED)
ADDER_CASES = ADDER_FAILURES = 0
for m in range(1, 13):
    for _ in range(10):
        a, b = rng.randrange(2 ** m), rng.randrange(2 ** m)
        s1, s2, carry = qdk.code.CheckAdders(m, a, b)
        ADDER_CASES += 1
        ADDER_FAILURES += not (s1 == s2 == (a + b) % 2 ** m and carry == (a + b >= 2 ** m))
assert ADDER_FAILURES == 0
for m in (8, 1062):
    assert qsharp.logical_counts(qdk.code.CountAdder, m)["cczCount"] == 2 * (m - 1)
print(f"adders: {ADDER_CASES - ADDER_FAILURES}/{ADDER_CASES} correct; 2(m - 1) Toffolis per m-qubit addition")

adders: 120/120 correct; 2(m - 1) Toffolis per m-qubit addition


### 2. The whole modular exponentiation, exactly

With the coset padding and the runways prepared as zero instead of in superposition, the circuit is an
exact classical computation, so its output can be compared bit for bit with `pow(g, e, N)`. The padding
follows the paper's formula, $c_\text{pad} = \lceil 2\lg n + \lg n_e\rceil + \delta$: carries absorbed by
a runway and the coset multiple both grow with every lookup addition, and with much less padding they
overflow (Sec. 2.9 bounds exactly this). The instances cover one, two and three runway pieces and a
partial exponent window. The simulator also stops with an error if any temporary qubit is released in
a state other than $|0\rangle$, so every uncomputation is checked too.

In [4]:
def random_modulus(n, rng):
    while True:
        N = rng.randrange(2 ** (n - 1) + 1, 2 ** n, 2)
        if N % 3 and N % 5:
            return N

EXACT_CASES = [  # (n, n_e, c_sep, c_exp, c_mul, delta)
    (24, 16, 24, 3, 3, 2),
    (48, 16, 24, 3, 2, 2),
    (64, 20, 32, 2, 3, 1),
    (72, 12, 24, 3, 3, 1),
    (50, 14, 25, 4, 2, 1),
]
rng = random.Random(SEED)
EXACT_TOTAL = EXACT_PASSED = 0
t0 = time.time()
for n, ne, csep, cexp, cmul, delta in EXACT_CASES:
    cpad = math.ceil(math.log2(n * n * ne)) + delta
    N = random_modulus(n, rng)
    g = rng.randrange(2, N)
    cfg = make_config(n, N, g, ne, cexp, cmul, csep, cpad, superpose=False)
    for _ in range(4):
        e = rng.randrange(2 ** ne)
        acc, work, _ = qdk.code.RunModExp(cfg, False, [bool((e >> b) & 1) for b in range(ne)])
        ok = decode(acc, n, csep, cpad) % N == pow(g, e, N) and decode(work, n, csep, cpad) % N == 0
        EXACT_TOTAL += 1
        EXACT_PASSED += ok
    print(f"n = {n:3d}, pieces = {-(-n // csep)}, c_pad = {cpad}: passed {EXACT_PASSED}/{EXACT_TOTAL} so far")
assert EXACT_PASSED == EXACT_TOTAL
print(f"exact modular exponentiation: {EXACT_PASSED}/{EXACT_TOTAL} correct ({time.time() - t0:.0f}s)")

n =  24, pieces = 1, c_pad = 16: passed 4/4 so far
n =  48, pieces = 2, c_pad = 18: passed 8/8 so far


n =  64, pieces = 2, c_pad = 18: passed 12/12 so far


n =  72, pieces = 3, c_pad = 17: passed 16/16 so far
n =  50, pieces = 2, c_pad = 17: passed 20/20 so far
exact modular exponentiation: 20/20 correct (26s)


### 3. Initialization in superposition

Every step of the circuit after initialization permutes computational basis states, so check 2 covers
its arithmetic. What it does not cover is preparing the coset padding and the runways in superposition
(Sec. 2.15). For the coset, each padding qubit in $|+\rangle$ controls adding $N 2^j$ and is then
uncomputed by the comparison $x \ge N 2^j$. For each runway, a uniform value $r$ is placed in the runway
and subtracted from the next piece, from the top piece down. Subtracting can wrap when the piece above is
smaller than $r$; that is part of the approximation the paper accepts, and with tiny paddings it is
frequent. The check prepares registers with tiny paddings, measures them, and compares the measured
failure rate with the exact rate, counted classically over every coset multiple and runway value.

In [5]:
def init_failure_probability(N, csep, cpad, value):
    import itertools
    n = N.bit_length()
    lens = piece_lens(n, csep)
    w = len(lens)
    fails = total = 0
    for j in range(2 ** cpad):
        x = value + j * N
        base = [(x >> (i * csep)) & ((1 << low) - 1) if i < w - 1 else x >> (i * csep) for i, low in enumerate(lens)]
        for rs in itertools.product(range(2 ** cpad), repeat=w - 1):
            pieces = list(base)
            for i in range(w - 2, -1, -1):
                pieces[i] += rs[i] << lens[i]
                pieces[i + 1] = (pieces[i + 1] - rs[i]) % (1 << (lens[i + 1] + cpad))
            total += 1
            fails += sum(p << (i * csep) for i, p in enumerate(pieces)) % N != value % N
    return fails / total

INIT_SHOTS = 200
INIT_CHECKS = {}
for N, csep, cpad in [(221, 4, 4), (437, 3, 3)]:
    n = N.bit_length()
    cfg = qdk.code.MakeConfig(n, csep, cpad, 2, [], [], N, True, False, False)
    shots = run(cfg, False, [], INIT_SHOTS)
    measured = sum(decode(acc, n, csep, cpad) % N != 1 for acc, _, _ in shots) / INIT_SHOTS
    exact = init_failure_probability(N, csep, cpad, 1)
    z = (measured - exact) / math.sqrt(exact * (1 - exact) / INIT_SHOTS)
    pieces = -(-n // csep)
    INIT_CHECKS[pieces] = (measured, exact, z)
    print(f"N = {N}, {pieces} pieces, c_pad = {cpad}: measured failure rate {measured:.3f}, exact {exact:.3f}, z = {z:+.1f}")
    assert abs(z) < 4

N = 221, 2 pieces, c_pad = 4: measured failure rate 0.060, exact 0.066, z = -0.4


N = 437, 3 pieces, c_pad = 3: measured failure rate 0.165, exact 0.174, z = -0.3


### 4. Order finding end to end: the phases

Checks 2 and 3 look only at computational basis states. Shor's algorithm also needs the circuit to keep
the exponent's superposition coherent: garbage left entangled with the exponent would wash out the
interference. This check runs order finding for $N = 21$, $g = 2$ (order 6) with $n_e = 2n$: the
exponent starts in superposition, the coset padding is in superposition, and each window is read out by
the semi-classical Fourier transform. The measured frequencies $y$ should land on the peaks of the ideal
distribution, computed classically, near multiples of $2^{n_e}/6$. The register is small enough to fit in one
runway piece, so this instance has no runways (their phases are covered by the fact that every runway
operation permutes basis states).

In [6]:
SHOR_N, SHOR_G, SHOR_CSEP, SHOR_CPAD, SHOR_CEXP, SHOR_CMUL, SHOR_SHOTS = 21, 2, 8, 6, 2, 2, 40
n = SHOR_N.bit_length()
ne = 2 * n
Q = 2 ** ne
order = next(k for k in range(1, SHOR_N) if pow(SHOR_G, k, SHOR_N) == 1)

ideal = np.zeros(Q)
exps = np.arange(Q)
for s in range(order):
    ideal += np.abs(np.fft.fft((exps % order == s).astype(float)) / Q) ** 2
peaks = ideal > 1e-3


def recovers_order(y):
    return Fraction(y, Q).limit_denominator(SHOR_N - 1).denominator == order


t0 = time.time()
cfg = make_config(n, SHOR_N, SHOR_G, ne, SHOR_CEXP, SHOR_CMUL, SHOR_CSEP, SHOR_CPAD, superpose=True)
ys = [sum(1 << k for k, bit in enumerate(freq) if bit) for _, _, freq in run(cfg, True, [], SHOR_SHOTS)]
SHOR = {
    "peak_mass": float(np.mean([peaks[y] for y in ys])),
    "peak_mass_ideal": float(ideal[peaks].sum()),
    "peak_mass_uniform": float(peaks.mean()),
    "order_recovered": float(np.mean([recovers_order(y) for y in ys])),
    "order_recovered_ideal": float(sum(ideal[y] for y in range(Q) if recovers_order(y))),
}
print(f"{SHOR_SHOTS} shots ({time.time() - t0:.0f}s)")
print(f"fraction of shots on the ideal peaks: {SHOR['peak_mass']:.3f}  (ideal circuit {SHOR['peak_mass_ideal']:.3f}, "
      f"uniform noise {SHOR['peak_mass_uniform']:.3f})")
print(f"order recovered by continued fractions: {SHOR['order_recovered']:.3f}  (ideal {SHOR['order_recovered_ideal']:.3f})")
assert SHOR["peak_mass"] > 0.8

40 shots (150s)
fraction of shots on the ideal peaks: 0.950  (ideal circuit 0.971, uniform noise 0.029)
order recovered by continued fractions: 0.250  (ideal 0.322)


## Counting the circuit at full size

The circuit is counted with the QDK's logical resource counter at $n = 2048$ with Table 3's parameters.
Lookup tables are all zeros when counting: the Toffoli count of a lookup does not depend on its contents
(the lookup's structure is fixed by its address size), and the modulus is a placeholder,
$2^{2047} + 1$, which affects only the initialization's constant additions.

Every windowed multiplication with the same window size has exactly the same structure. Counting all
606 of them directly runs out of memory in the QDK's estimate caching, so the count is composed: the
circuit with no windows (initialization and readout), the circuit with one window of each size that
occurs, and the number of windows of each size. Lookup additions inside a window are traced once per
address size with estimate caching. The first cell checks on three smaller instances that the composed
count equals the count of the whole circuit, with and without caching.

In [7]:
ADDITIVE = ("cczCount", "ccixCount", "tCount", "rotationCount", "measurementCount")


def count_windows(n, csep, cpad, cmul, windows, cache=True):
    cfg = qdk.code.MakeConfig(n, csep, cpad, cmul, windows, [], 2 ** (n - 1) + 1, True, True, cache)
    return qsharp.logical_counts(qdk.code.CountModExp, cfg)


def composed_count(n, ne, csep, cpad, cexp, cmul):
    sizes = [hi - lo for lo, hi in exponent_windows(ne, cexp)]
    base = count_windows(n, csep, cpad, cmul, [])
    per_window = {c: count_windows(n, csep, cpad, cmul, [c]) for c in sorted(set(sizes))}
    multiplicity = Counter(sizes)
    total = {k: base[k] + sum(multiplicity[c] * (per_window[c][k] - base[k]) for c in per_window) for k in ADDITIVE}
    total["numQubits"] = max(c["numQubits"] for c in per_window.values())
    return total, base, per_window, multiplicity


COMPOSITION_CHECKS = 0
for n, ne, csep, cache in [(96, 22, 48, False), (256, 162, 128, True), (1024, 102, 512, True)]:
    cpad = math.ceil(2 * math.log2(n) + math.log2(ne)) + 4
    direct = count_windows(n, csep, cpad, 5, [hi - lo for lo, hi in exponent_windows(ne, 5)], cache)
    composed, *_ = composed_count(n, ne, csep, cpad, 5, 5)
    assert all(direct[k] == composed[k] for k in ADDITIVE + ("numQubits",)), (n, direct, composed)
    COMPOSITION_CHECKS += 1
    print(f"n = {n}, n_e = {ne}: composed count equals the direct {'cached' if cache else 'uncached'} count")

n = 96, n_e = 22: composed count equals the direct uncached count


n = 256, n_e = 162: composed count equals the direct cached count


n = 1024, n_e = 102: composed count equals the direct cached count


In [8]:
t0 = time.time()
COUNTS, BASE_COUNTS, WINDOW_COUNTS, WINDOW_MULTIPLICITY = composed_count(N_BITS, N_E, C_SEP, C_PAD, C_EXP, C_MUL)
TOFFOLIS = COUNTS["cczCount"] + COUNTS["ccixCount"]
LOGICAL_QUBITS = COUNTS["numQubits"]
print(f"counted in {time.time() - t0:.0f}s: windows {dict(WINDOW_MULTIPLICITY)}")
print(f"Toffolis per run: {TOFFOLIS:,}   peak logical qubits: {LOGICAL_QUBITS:,}")
print(COUNTS)

counted in 5s: windows {4: 1, 5: 605}
Toffolis per run: 2,693,430,190   peak logical qubits: 6,430
{'cczCount': 2693430190, 'ccixCount': 0, 'tCount': 0, 'rotationCount': 3029, 'measurementCount': 1580249027, 'numQubits': 6430}


### Where the Toffolis go

The timing model needs the circuit's structure: how many lookup additions there are, how large each
lookup's address is, and how long each adder's carry chain is. The cell tallies them from the same loop
structure as `modexp.qs`, prices each piece with a separate Q# count, and checks that the tally adds up
to the full count exactly.

In [9]:
FACTOR_BITS = N_BITS + (PIECES - 1) + C_PAD          # Fig. 3: each lower runway reduced to one qubit
CHUNKS = [C_MUL] * (FACTOR_BITS // C_MUL) + ([FACTOR_BITS % C_MUL] if FACTOR_BITS % C_MUL else [])
WINDOW_SIZES = [hi - lo for lo, hi in exponent_windows(N_E, C_EXP)]
LOOKUP_ADDITIONS = Counter(c + chunk for c in WINDOW_SIZES for _ in range(2) for chunk in CHUNKS)  # address size -> count
MULTIPLY_ADDS = 2 * len(WINDOW_SIZES)

LOOKUP_TOFFOLIS = {a: qsharp.logical_counts(qdk.code.CountLookup, N_BITS, a)["cczCount"] for a in LOOKUP_ADDITIONS}
UNLOOKUP_TOFFOLIS = {a: qsharp.logical_counts(qdk.code.CountLookupUnlookup, N_BITS, a)["cczCount"] - LOOKUP_TOFFOLIS[a]
                     for a in LOOKUP_ADDITIONS}
ADDER_TOFFOLIS = qsharp.logical_counts(qdk.code.CountAdder, C_SEP + C_PAD)["cczCount"]   # one runway piece
REDUCTION_TOFFOLIS = 2 * (2 * C_PAD) * (PIECES - 1)   # reduce and restore, per multiply-add
INIT_TOFFOLIS = BASE_COUNTS["cczCount"]

tally = (INIT_TOFFOLIS
         + sum(k * (LOOKUP_TOFFOLIS[a] + UNLOOKUP_TOFFOLIS[a] + PIECES * ADDER_TOFFOLIS) for a, k in LOOKUP_ADDITIONS.items())
         + MULTIPLY_ADDS * REDUCTION_TOFFOLIS)
assert tally == TOFFOLIS, (tally, TOFFOLIS)
N_LOOKUP_ADDITIONS = sum(LOOKUP_ADDITIONS.values())
print(f"lookup additions per run: {N_LOOKUP_ADDITIONS:,}  by address size {dict(sorted(LOOKUP_ADDITIONS.items()))}")
print(f"per lookup addition (10 address bits): lookup {LOOKUP_TOFFOLIS[10]}, unlookup {UNLOOKUP_TOFFOLIS[10]}, "
      f"adders {PIECES} x {ADDER_TOFFOLIS}")
print(f"initialization {INIT_TOFFOLIS:,} Toffolis ({INIT_TOFFOLIS / TOFFOLIS:.2e} of the total); tally matches the count")

lookup additions per run: 506,616  by address size {6: 2, 7: 1210, 9: 834, 10: 504570}
per lookup addition (10 address bits): lookup 1022, unlookup 52, adders 2 x 2122
initialization 632,534 Toffolis (2.35e-04 of the total); tally matches the count


### The paper's asymptotic expressions

For comparison only, the abstract's expressions evaluated at $n = 2048$ (they assume $n_e = 1.5n$ and
leave out distillation and routing). The paper does not print these values; they are shown here for
comparison only and are not part of `results.json`.

In [10]:
LG = math.log2(N_BITS)
ABSTRACT = {
    "logical_qubits": 3 * N_BITS + 0.002 * N_BITS * LG,
    "toffoli_count": 0.3 * N_BITS ** 3 + 0.0005 * N_BITS ** 3 * LG,
    "measurement_depth": 500 * N_BITS ** 2 + N_BITS ** 2 * LG,
}
for k, v in ABSTRACT.items():
    print(f"{k:18s} {v:,.0f}")
print(f"circuit / expression: logical qubits {LOGICAL_QUBITS / ABSTRACT['logical_qubits']:.3f}, "
      f"Toffolis {TOFFOLIS / ABSTRACT['toffoli_count']:.3f}")

logical_qubits     6,189
toffoli_count      2,624,225,018
measurement_depth  2,143,289,344
circuit / expression: logical qubits 1.039, Toffolis 1.026


## Cross-check: the authors' cost script

Tables 3–5 and Fig. 1 were produced by the ancillary file `estimate_costs.py`. The cell downloads it
from arXiv, checks it against a pinned SHA-256, and runs it twice: once at Table 3's parameters, and once
through its own parameter search for RSA-2048, which should select Table 3's row. It needs only
matplotlib (for its plotting functions, which are not called). Nothing it returns enters QAC's figures
or `results.json`; its values appear only in the output below.

In [11]:
import hashlib
import importlib.util
import os
import tempfile
import urllib.request

AUTHOR_SCRIPT_URL = "https://arxiv.org/src/1905.09749v3/anc/estimate_costs.py"
AUTHOR_SCRIPT_SHA256 = "f5a6464cf8f319c89adade8ad368efe15bb6980b22d963a24ba6db5ee3537ea0"

os.environ["MPLBACKEND"] = "Agg"
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "estimate_costs.py"
    with urllib.request.urlopen(AUTHOR_SCRIPT_URL, timeout=60) as response:
        path.write_bytes(response.read())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == AUTHOR_SCRIPT_SHA256, "author script changed"
    spec = importlib.util.spec_from_file_location("gidney_ekera_estimate_costs", path)
    author = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(author)

import datetime
table3_params = author.Parameters(
    gate_err=PHYSICAL_ERROR_RATE,
    reaction_time=datetime.timedelta(microseconds=REACTION_TIME_NS / 1000),
    cycle_time=datetime.timedelta(microseconds=CYCLE_TIME_NS / 1000),
    exp_window=C_EXP, mul_window=C_MUL, runway_sep=C_SEP,
    l1_distance=LEVEL1_DISTANCE_TABLE3, code_distance=LEVEL2_DISTANCE_TABLE3,
    max_total_err=0.8, n=N_BITS, n_e=N_E, use_t_t_distillation=False, deviation_padding=C_PAD)
AUTHOR = author.estimate_algorithm_cost(table3_params)
t0 = time.time()
AUTHOR_SEARCH = author.eh_rsa(N_BITS, PHYSICAL_ERROR_RATE)
p = AUTHOR_SEARCH.params
AUTHOR_SEARCH_MATCHES_TABLE3 = (
    (p.l1_distance, p.code_distance, p.mul_window, p.exp_window, p.runway_sep, p.n_e, p.deviation_padding)
    == (LEVEL1_DISTANCE_TABLE3, LEVEL2_DISTANCE_TABLE3, C_MUL, C_EXP, C_SEP, N_E, C_PAD))
print(f"author script at Table 3's parameters: {AUTHOR.toffoli_count:,.0f} Toffolis, {AUTHOR.total_megaqubits:.2f} megaqubits, "
      f"{AUTHOR.total_hours:.2f} h per run, retry risk {AUTHOR.total_error:.0%}, "
      f"expected {AUTHOR.total_hours / (1 - AUTHOR.total_error):.2f} h")
print(f"its own search ({time.time() - t0:.0f}s) selects d1 = {p.l1_distance}, d2 = {p.code_distance}, "
      f"c_mul = {p.mul_window}, c_exp = {p.exp_window}, c_sep = {p.runway_sep}, c_pad = {p.deviation_padding}: "
      f"{'Table 3' if AUTHOR_SEARCH_MATCHES_TABLE3 else 'NOT Table 3'}")
print(f"circuit / author script: Toffolis {TOFFOLIS / AUTHOR.toffoli_count:.4f}")

author script at Table 3's parameters: 2,698,295,530 Toffolis, 19.25 megaqubits, 5.05 h per run, retry risk 31%, expected 7.31 h
its own search (0s) selects d1 = 15, d2 = 27, c_mul = 5, c_exp = 5, c_sep = 1024, c_pad = 38: Table 3
circuit / author script: Toffolis 0.9982


## Physical cost: the paper's layout and timing, rebuilt

`physical_layout.py` implements Secs. 2.10–2.14 with each constant and its locator, and the cell
applies it to the circuit's counts:

- **Layout (Sec. 2.10).** Each runway piece is 113 logical qubits wide (fourteen CCZ factories, each
  15 × 8 at level-1 distance 17, in two rows with gaps) and 33 rows of operating area tall, plus the
  rows that the three registers' share of the piece ($c_\text{sep} + c_\text{pad}$ qubits each) fills at
  that width. Each patch takes $2(d+1)^2$ physical qubits at $d = 27$ (Sec. 2.14). The text states the
  factory footprint only for level-1 distance 17, so this layout is the one Sec. 2.14 evaluates (about
  23 million qubits there), not the script's layout at distance 15.
- **Time (Sec. 2.11).** Per lookup addition: the lookup is code-depth limited at $d/2$ cycles per
  sequential Toffoli (double-access hallways); the addition is reaction limited, one reaction time per
  Toffoli of one piece's adder (pieces run in parallel); and about 1 ms for the rest. The runway
  reductions are reaction-limited additions too.
- **Retries (Secs. 2.9, 2.12, 2.13, 3.1).** The topological error is $10^{-\lceil d/2+1\rceil}$ per
  patch per cycle over the non-factory patches; the distillation error scales Sec. 2.12's 6.4% for about
  3 billion CCZ states (the text gives no per-state figure) to the circuit's Toffoli count; the
  approximation error is $2\sqrt{\text{deviation}}$; post-processing fails with probability 1%. They
  combine assuming independence, and the expected runtime is the time per run divided by the chance a
  run succeeds.

In [12]:
import physical_layout as pl

LAYOUT = pl.layout(pieces=PIECES, piece_qubits=C_SEP + C_PAD, d=pl.CODE_DISTANCE)

time_per_run = (sum(k * (pl.lookup_time_s(LOOKUP_TOFFOLIS[a]) + pl.addition_time_s(ADDER_TOFFOLIS)
                         + pl.OTHER_LOOKUP_ADDITION_TIME_S) for a, k in LOOKUP_ADDITIONS.items())
                + MULTIPLY_ADDS * pl.runway_reduction_time_s(2 * 2 * C_PAD))   # reduce and restore; pieces run in parallel
cycles = time_per_run / pl.CYCLE_TIME_S
ERRORS = {
    "topological": pl.topological_error(LAYOUT["logical_patches"] - LAYOUT["factory_patches"], cycles),
    "distillation": pl.distillation_error(TOFFOLIS),
    "approximation": pl.approximation_error(N_LOOKUP_ADDITIONS, N_BITS, C_SEP, C_PAD),
    "post_processing": pl.POST_PROCESSING_FAILURE,
}
RETRY_RISK = pl.union(*ERRORS.values())
EXPECTED_RUNS = 1 / (1 - RETRY_RISK)
PAPER_LAYOUT = {
    **LAYOUT,
    "seconds_per_run": time_per_run,
    "runtime_seconds": time_per_run * EXPECTED_RUNS,
}
print(f"board {LAYOUT['board_width']} x {LAYOUT['board_height']} = {LAYOUT['logical_patches']:,} patches "
      f"({LAYOUT['factory_patches'] / LAYOUT['logical_patches']:.1%} factories) -> {LAYOUT['physical_qubits']:,} physical qubits")
print(f"per run {time_per_run / 3600:.2f} h; errors " + ", ".join(f"{k} {v:.3g}" for k, v in ERRORS.items()))
print(f"retry risk {RETRY_RISK:.1%}, expected runs {EXPECTED_RUNS:.3f}, expected runtime {PAPER_LAYOUT['runtime_seconds'] / 3600:.2f} h")

board 226 x 63 = 14,238 patches (23.6% factories) -> 22,325,184 physical qubits
per run 5.06 h; errors topological 0.198, distillation 0.0575, approximation 0.00384, post_processing 0.01
retry risk 25.5%, expected runs 1.342, expected runtime 6.79 h


## Physical cost: an independent check with the QDK resource estimator

The estimator is handed the circuit's counts as a trace: one run is `TOFFOLIS` Toffoli gates on
`LOGICAL_QUBITS` logical qubits. It models what it can of the paper's assumptions: a gate-based
architecture with a $10^{-3}$ error rate, surface codes with a 1 µs cycle, distances searched over
25–51 as the paper's search does (Sec. 3.1), and Toffolis consumed as magic states through lattice
surgery. It has no model of the paper's CCZ factory, so Toffoli states come from magic state cultivation
followed by the 8T-to-CCX factory of Gidney, Shutty, and Jones (2024). It also has no reaction time and
schedules the Toffolis one after another, without the paper's parallel runway pieces.

The per-run error budget is the one the rebuilt layout spends on topological and distillation errors.
Among the Pareto-optimal estimates, the one minimizing the paper's objective
$s^{1.2} t / (1 - \epsilon)$ (Sec. 3.1) is kept; the expected runtime divides by the chance that a run
succeeds, including the approximation and post-processing failures.

In [13]:
import re
from qdk.qre import Application, Trace, PSSPC, LatticeSurgery, estimate, instruction_ids
from qdk.qre.models import GateBased, SurfaceCode, GSJ24Factory, GSJ24CCXFactory


class FactoringRun(Application[None]):
    def get_trace(self, parameters: None) -> Trace:
        trace = Trace(LOGICAL_QUBITS)
        block = trace.add_block(repetitions=TOFFOLIS)
        block.add_operation(instruction_ids.CCX, [0, 1, 2])
        return trace


QRE_ARCHITECTURE = dict(error_rate=PHYSICAL_ERROR_RATE, gate_time=100, measurement_time=500)   # ns
QRE_DISTANCES = list(range(25, 52, 2))
QRE_BUDGET = pl.union(ERRORS["topological"], ERRORS["distillation"])
qre = estimate(
    FactoringRun(), GateBased(**QRE_ARCHITECTURE),
    SurfaceCode.q(distance=QRE_DISTANCES, code_cycle_override=CYCLE_TIME_NS) * GSJ24Factory.q() * GSJ24CCXFactory.q(),
    PSSPC.q(ccx_magic_states=True) * LatticeSurgery.q(),
    max_error=QRE_BUDGET, name="Gidney-Ekera RSA-2048")
qre.add_factory_summary_column()
qre.add_qubit_partition_column()
frame = qre.as_frame()
display(frame)


def objective(i):
    row = frame.iloc[i]
    return row["qubits"] ** 1.2 * row["runtime"].total_seconds() / (1 - row["error"])


best = min(range(len(frame)), key=objective)
row = frame.iloc[best]
QRE_DISTANCE = int(re.search(r"^LATTICE_SURGERY @ SurfaceCode\(.*distance=(\d+)\)", str(qre[best].source), re.MULTILINE).group(1))
QRE_RETRY = pl.union(float(row["error"]), ERRORS["approximation"], ERRORS["post_processing"])
QRE = {
    "physical_qubits": int(row["qubits"]),
    "compute_qubits": int(row["physical_compute_qubits"]),
    "factory_qubits": int(row["physical_factory_qubits"]),
    "memory_qubits": int(row["physical_memory_qubits"]),
    "factories": str(row["factories"]),
    "seconds_per_run": float(row["runtime"].total_seconds()),
    "error": float(row["error"]),
    "expected_runs": 1 / (1 - QRE_RETRY),
}
QRE["runtime_seconds"] = QRE["seconds_per_run"] * QRE["expected_runs"]
print(f"selected distance {QRE_DISTANCE}; {QRE['physical_qubits']:,} physical qubits; {QRE['seconds_per_run'] / 3600:.1f} h per run; "
      f"expected {QRE['runtime_seconds'] / 3600:.1f} h")

,name,qubits,runtime,error,factories,physical_compute_qubits,physical_factory_qubits,physical_memory_qubits
0,Gidney-Ekera RSA-2048,19099192,2 days 12:36:07.845390,0.067819,2×CCX,19069216,29976,0


selected distance 27; 19,099,192 physical qubits; 60.6 h per run; expected 65.9 h


## Results

The rebuilt layout is the headline physical scenario, since it is the paper's own model applied to the
circuit's own counts; the estimator's scenario follows it. Figures per run are kept alongside the
expected figures that include repeated runs.

In [14]:
import json
import platform
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata

CLAIM_ID = "gidney-ekera-rsa2021"
REPRODUCTION_ID = "qsharp-modexp-qre-2026-11"


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "unknown"


results_record = {
    "schema_version": "1.0",
    "claim_id": CLAIM_ID,
    "reproduction_id": REPRODUCTION_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "producer": {"entrypoint": "notebook.ipynb", "cell_tag": "results-export"},
    "tooling": {
        "python": platform.python_version(),
        "packages": {name: package_version(name) for name in ("qdk", "numpy", "pandas", "matplotlib")},
    },
    "inputs": {
        "factoring": {
            "kind": "rsa",
            "modulus_bits": N_BITS,
            "algorithm": "Ekerå–Håstad short discrete logarithm, one run; windowed modular exponentiation in the coset representation with oblivious carry runways",
            "parameters": {
                "n_e": N_E,
                "c_exp": C_EXP,
                "c_mul": C_MUL,
                "c_sep": C_SEP,
                "c_pad": C_PAD,
                "delta_off": DELTA_OFF,
                "level1_distance_table3": LEVEL1_DISTANCE_TABLE3,
                "level2_distance_table3": LEVEL2_DISTANCE_TABLE3,
                "count_modulus": "2^2047 + 1 (placeholder; lookup tables are all zeros when counting)",
            },
        }
    },
    "logical": {
        "num_qubits": int(LOGICAL_QUBITS),
        "t_count": int(COUNTS["tCount"]),
        "rotation_count": int(COUNTS["rotationCount"]),
        "ccz_count": int(COUNTS["cczCount"]),
        "ccix_count": int(COUNTS["ccixCount"]),
        "measurement_count": int(COUNTS["measurementCount"]),
        "toffoli_count": int(COUNTS["cczCount"]) + int(COUNTS["ccixCount"]),
    },
    "physical": [
        {
            "scenario_id": "paper-layout-rebuilt",
            # The schema requires gate and measurement times; the paper states only the cycle and reaction
            # times, and this model uses only those. The estimator's gate and measurement times are recorded.
            "architecture": {"physical_error_rate": PHYSICAL_ERROR_RATE, "gate_time_ns": QRE_ARCHITECTURE["gate_time"],
                             "measurement_time_ns": QRE_ARCHITECTURE["measurement_time"]},
            "code": {"family": "surface", "distance": pl.CODE_DISTANCE, "distance_source": "imposed",
                     "cycle_time_ns": CYCLE_TIME_NS},
            "factories": f"{LAYOUT['factories']} CCZ factories (Gidney and Fowler), {pl.FACTORY_WIDTH} x {pl.FACTORY_HEIGHT} logical patches each at level-1 distance {pl.LEVEL1_DISTANCE} (Sec. 2.10)",
            "synthesis": f"Toffolis as CCZ states converted to AutoCCZ states; reaction-limited ripple-carry additions in parallel runway pieces ({REACTION_TIME_NS} ns reaction time); code-depth-limited lookups with double-access hallways (Secs. 2.10-2.11)",
            "physical_qubits": int(LAYOUT["physical_qubits"]),
            "compute_qubits": int(LAYOUT["compute_qubits"]),
            "factory_qubits": int(LAYOUT["factory_qubits"]),
            "memory_qubits": 0,
            "runtime_seconds": round(PAPER_LAYOUT["runtime_seconds"], 3),
            "runtime_seconds_per_shot": round(PAPER_LAYOUT["seconds_per_run"], 3),
            "error_budget": 1.0,
            "achieved_error": round(RETRY_RISK, 6),
        },
        {
            "scenario_id": "qre-cultivation-ccx",
            "architecture": {"physical_error_rate": PHYSICAL_ERROR_RATE, "gate_time_ns": QRE_ARCHITECTURE["gate_time"],
                             "measurement_time_ns": QRE_ARCHITECTURE["measurement_time"]},
            "code": {"family": "surface", "distance": QRE_DISTANCE, "distance_source": "selected_by_tool",
                     "cycle_time_ns": CYCLE_TIME_NS},
            "factories": QRE["factories"],
            "synthesis": "Toffolis as CCX magic states (PSSPC) from magic state cultivation and the GSJ24 8T-to-CCX factory; lattice surgery; Toffolis executed one after another",
            "physical_qubits": QRE["physical_qubits"],
            "compute_qubits": QRE["compute_qubits"],
            "factory_qubits": QRE["factory_qubits"],
            "memory_qubits": QRE["memory_qubits"],
            "runtime_seconds": round(QRE["runtime_seconds"], 3),
            "runtime_seconds_per_shot": round(QRE["seconds_per_run"], 3),
            "error_budget": round(QRE_BUDGET, 6),
            "achieved_error": round(QRE["error"], 6),
        },
    ],
}

with open("results.json", "w", encoding="utf-8") as handle:
    json.dump(results_record, handle, indent=2, ensure_ascii=False)
    handle.write("\n")

print(json.dumps(results_record["logical"], indent=2))
print({s["scenario_id"]: (s["physical_qubits"], round(s["runtime_seconds"] / 3600, 2)) for s in results_record["physical"]})

{
  "num_qubits": 6430,
  "t_count": 0,
  "rotation_count": 3029,
  "ccz_count": 2693430190,
  "ccix_count": 0,
  "measurement_count": 1580249027,
  "toffoli_count": 2693430190
}
{'paper-layout-rebuilt': (22325184, 6.79), 'qre-cultivation-ccx': (19099192, 65.92)}
